In [0]:
telemetry = spark.table("workspace.default.silver_telemetry")
errors = spark.table("workspace.default.silver_errors")
failures = spark.table("workspace.default.silver_failures")

print("Telemetry Schema:")
telemetry.printSchema()

print("Errors Schema:")
errors.printSchema()

print("Failures Schema:")
failures.printSchema()

Telemetry Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)

Errors Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- errorID: string (nullable = true)

Failures Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- failure: string (nullable = true)



In [0]:
from pyspark.sql.functions import count

component_counts = (
    failures
    .groupBy("failure")
    .agg(count("*").alias("failure_count"))
    .orderBy("failure")
)

component_counts.show()

+-------+-------------+
|failure|failure_count|
+-------+-------------+
|  comp1|          192|
|  comp2|          259|
|  comp3|          131|
|  comp4|          179|
+-------+-------------+



In [0]:
from pyspark.sql.functions import col, max

failure_telemetry = (
    failures.alias("f")
    .join(
        telemetry.alias("t"),
        (col("f.machineID") == col("t.machineID")) &
        (col("t.datetime") <= col("f.datetime")),
        "left"
    )
    .groupBy(
        col("f.datetime").alias("failure_datetime"),
        col("f.machineID").alias("machineID"),
        col("f.failure").alias("failure")
    )
    .agg(
        max("t.datetime").alias("telemetry_datetime")
    )
)

print("Failure events with latest telemetry timestamp:")
print("Records:", failure_telemetry.count())

failure_telemetry.show(10, truncate=False)

Failure events with latest telemetry timestamp:
Records: 761
+-------------------+---------+-------+-------------------+
|failure_datetime   |machineID|failure|telemetry_datetime |
+-------------------+---------+-------+-------------------+
|2015-11-12 06:00:00|34       |comp1  |2015-11-12 06:00:00|
|2015-04-19 06:00:00|37       |comp4  |2015-04-19 06:00:00|
|2015-11-20 06:00:00|47       |comp2  |2015-11-20 06:00:00|
|2015-10-28 06:00:00|34       |comp2  |2015-10-28 06:00:00|
|2015-09-16 06:00:00|37       |comp2  |2015-09-16 06:00:00|
|2015-12-20 06:00:00|47       |comp2  |2015-12-20 06:00:00|
|2015-04-16 06:00:00|34       |comp1  |2015-04-16 06:00:00|
|2015-07-18 06:00:00|37       |comp4  |2015-07-18 06:00:00|
|2015-04-09 06:00:00|47       |comp2  |2015-04-09 06:00:00|
|2015-05-19 06:00:00|37       |comp3  |2015-05-19 06:00:00|
+-------------------+---------+-------+-------------------+
only showing top 10 rows


In [0]:
failure_features = (
    failure_telemetry.alias("f")
    .join(
        telemetry.alias("t"),
        (col("f.machineID") == col("t.machineID")) &
        (col("f.telemetry_datetime") == col("t.datetime")),
        "inner"
    )
    .select(
        col("f.failure_datetime"),
        col("f.machineID"),
        col("f.failure"),
        col("t.volt"),
        col("t.rotate"),
        col("t.pressure"),
        col("t.vibration")
    )
)

print("Failure feature records:", failure_features.count())

print("Failure feature schema:")
failure_features.printSchema()

failure_features.show(10, truncate=False)

Failure feature records: 761
Failure feature schema:
root
 |-- failure_datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- failure: string (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)

+-------------------+---------+-------+----------------+----------------+----------------+----------------+
|failure_datetime   |machineID|failure|volt            |rotate          |pressure        |vibration       |
+-------------------+---------+-------+----------------+----------------+----------------+----------------+
|2015-10-17 06:00:00|1        |comp2  |178.322427612404|383.71525553    |79.704007684226 |43.2134165642225|
|2015-11-28 06:00:00|16       |comp2  |170.995646723153|327.3482711533  |126.586689062364|36.7456433315164|
|2015-01-02 03:00:00|17       |comp4  |158.098828106012|406.325390520277|97.0017647740545|59.9671740225826|
|2015-

In [0]:
from pyspark.sql.functions import collect_set, expr

failure_with_errors = (
    failure_features.alias("f")
    .join(
        errors.alias("e"),
        (col("f.machineID") == col("e.machineID")) &
        (col("e.datetime") < col("f.failure_datetime")) &
        (col("e.datetime") >= col("f.failure_datetime") - expr("INTERVAL 48 HOURS")),
        "left"
    )
    .groupBy(
        "f.failure_datetime",
        "f.machineID",
        "f.failure",
        "f.volt",
        "f.rotate",
        "f.pressure",
        "f.vibration"
    )
    .agg(
        collect_set("e.errorID").alias("error_ids")
    )
)

print("Failure records with error information:", failure_with_errors.count())

failure_with_errors.printSchema()

failure_with_errors.show(10, truncate=False)

Failure records with error information: 761
root
 |-- failure_datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- failure: string (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- error_ids: array (nullable = false)
 |    |-- element: string (containsNull = false)

+-------------------+---------+-------+----------------+----------------+----------------+----------------+------------------------+
|failure_datetime   |machineID|failure|volt            |rotate          |pressure        |vibration       |error_ids               |
+-------------------+---------+-------+----------------+----------------+----------------+----------------+------------------------+
|2015-12-22 06:00:00|13       |comp2  |154.864450029914|297.989464352549|93.2564186620334|46.6329034818821|[error1, error2, error3]|
|2015-12-22 06:00:00|13       |comp1  |15

In [0]:
from pyspark.sql.functions import array_contains, when, lit

model_data = (
    failure_with_errors
    .withColumn(
        "has_error1",
        when(array_contains("error_ids", "error1"), 1).otherwise(0)
    )
    .withColumn(
        "has_error2",
        when(array_contains("error_ids", "error2"), 1).otherwise(0)
    )
    .withColumn(
        "has_error3",
        when(array_contains("error_ids", "error3"), 1).otherwise(0)
    )
    .withColumn(
        "has_error4",
        when(array_contains("error_ids", "error4"), 1).otherwise(0)
    )
    .withColumn(
        "has_error5",
        when(array_contains("error_ids", "error5"), 1).otherwise(0)
    )
)

print("Model dataset created.")
print("Records:", model_data.count())

model_data.printSchema()

model_data.select(
    "machineID",
    "failure",
    "volt",
    "rotate",
    "pressure",
    "vibration",
    "has_error1",
    "has_error2",
    "has_error3",
    "has_error4",
    "has_error5"
).show(10, truncate=False)

Model dataset created.
Records: 761
root
 |-- failure_datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- failure: string (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- error_ids: array (nullable = false)
 |    |-- element: string (containsNull = false)
 |-- has_error1: integer (nullable = false)
 |-- has_error2: integer (nullable = false)
 |-- has_error3: integer (nullable = false)
 |-- has_error4: integer (nullable = false)
 |-- has_error5: integer (nullable = false)

+---------+-------+----------------+----------------+----------------+----------------+----------+----------+----------+----------+----------+
|machineID|failure|volt            |rotate          |pressure        |vibration       |has_error1|has_error2|has_error3|has_error4|has_error5|
+---------+-------+----------------+----------------+----------------+-----

In [0]:
model_pd = (
    model_data
    .select(
        "failure_datetime",
        "machineID",
        "failure",
        "volt",
        "rotate",
        "pressure",
        "vibration",
        "has_error1",
        "has_error2",
        "has_error3",
        "has_error4",
        "has_error5"
    )
    .toPandas()
)

print("Pandas dataset created.")
print("Records:", len(model_pd))
print("Machines:", model_pd["machineID"].nunique())

print("\nFailure class distribution:")
print(model_pd["failure"].value_counts())

print("\nDataset columns:")
print(model_pd.columns.tolist())

Pandas dataset created.
Records: 761
Machines: 98

Failure class distribution:
failure
comp2    259
comp1    192
comp4    179
comp3    131
Name: count, dtype: int64

Dataset columns:
['failure_datetime', 'machineID', 'failure', 'volt', 'rotate', 'pressure', 'vibration', 'has_error1', 'has_error2', 'has_error3', 'has_error4', 'has_error5']


In [0]:
model_pd = model_pd.sort_values(
    "failure_datetime"
).reset_index(drop=True)

split_index = int(len(model_pd) * 0.8)

train_pd = model_pd.iloc[:split_index].copy()
test_pd = model_pd.iloc[split_index:].copy()

print("Training records:", len(train_pd))
print("Testing records:", len(test_pd))

print("\nTraining period:")
print(train_pd["failure_datetime"].min(), "to", train_pd["failure_datetime"].max())

print("\nTesting period:")
print(test_pd["failure_datetime"].min(), "to", test_pd["failure_datetime"].max())

print("\nTraining class distribution:")
print(train_pd["failure"].value_counts())

print("\nTesting class distribution:")
print(test_pd["failure"].value_counts())

Training records: 608
Testing records: 153

Training period:
2015-01-02 03:00:00 to 2015-10-15 06:00:00

Testing period:
2015-10-17 06:00:00 to 2015-12-31 06:00:00

Training class distribution:
failure
comp2    201
comp1    158
comp4    142
comp3    107
Name: count, dtype: int64

Testing class distribution:
failure
comp2    58
comp4    37
comp1    34
comp3    24
Name: count, dtype: int64


In [0]:
feature_columns = [
    "volt",
    "rotate",
    "pressure",
    "vibration",
    "has_error1",
    "has_error2",
    "has_error3",
    "has_error4",
    "has_error5"
]

X_train = train_pd[feature_columns]
y_train = train_pd["failure"]

X_test = test_pd[feature_columns]
y_test = test_pd["failure"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nTarget classes:")
print(sorted(y_train.unique()))

X_train shape: (608, 9)
y_train shape: (608,)
X_test shape: (153, 9)
y_test shape: (153,)

Target classes:
['comp1', 'comp2', 'comp3', 'comp4']


In [0]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Random Forest multi-class model trained successfully.")
print("Classes:", model.classes_)

Random Forest multi-class model trained successfully.
Classes: ['comp1' 'comp2' 'comp3' 'comp4']


In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_test_prediction = model.predict(X_test)

accuracy = accuracy_score(y_test, y_test_prediction)
precision = precision_score(
    y_test,
    y_test_prediction,
    average="weighted",
    zero_division=0
)
recall = recall_score(
    y_test,
    y_test_prediction,
    average="weighted",
    zero_division=0
)
f1 = f1_score(
    y_test,
    y_test_prediction,
    average="weighted",
    zero_division=0
)

print("Q7 Model Evaluation")
print("------------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

print("\nClassification Report:")
print(classification_report(y_test, y_test_prediction, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_prediction))

Q7 Model Evaluation
------------------------
Accuracy : 0.9411764705882353
Precision: 0.9464190953139042
Recall   : 0.9411764705882353
F1-score : 0.9412081579813891

Classification Report:
              precision    recall  f1-score   support

       comp1       1.00      0.85      0.92        34
       comp2       0.96      0.95      0.96        58
       comp3       0.96      0.96      0.96        24
       comp4       0.86      1.00      0.93        37

    accuracy                           0.94       153
   macro avg       0.95      0.94      0.94       153
weighted avg       0.95      0.94      0.94       153

Confusion Matrix:
[[29  2  1  2]
 [ 0 55  0  3]
 [ 0  0 23  1]
 [ 0  0  0 37]]


In [0]:
# Get probability for each component
y_test_probabilities = model.predict_proba(X_test)

probability_columns = [
    "prob_comp1",
    "prob_comp2",
    "prob_comp3",
    "prob_comp4"
]

test_probability_df = test_pd[
    ["failure_datetime", "machineID", "failure"]
].copy()

for i, column in enumerate(probability_columns):
    test_probability_df[column] = y_test_probabilities[:, i]

# Predicted component
test_probability_df["predicted_component"] = model.classes_[
    y_test_probabilities.argmax(axis=1)
]

print(test_probability_df.head(10))

       failure_datetime  machineID  ... prob_comp4  predicted_component
608 2015-10-17 06:00:00         69  ...   0.986447                comp4
609 2015-10-17 06:00:00         59  ...   0.000000                comp2
610 2015-10-17 06:00:00         64  ...   0.000000                comp2
611 2015-10-17 06:00:00         23  ...   0.976455                comp4
612 2015-10-17 06:00:00          1  ...   0.000000                comp2
613 2015-10-17 06:00:00         95  ...   0.967633                comp4
614 2015-10-18 06:00:00         26  ...   0.000000                comp2
615 2015-10-18 06:00:00          8  ...   0.990000                comp4
616 2015-10-19 06:00:00         45  ...   1.000000                comp4
617 2015-10-19 06:00:00         31  ...   0.002930                comp2

[10 rows x 8 columns]


In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

latest_window = Window.partitionBy("machineID").orderBy(col("datetime").desc())

latest_telemetry = (
    telemetry
    .withColumn("rn", row_number().over(latest_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Latest telemetry records:", latest_telemetry.count())
latest_telemetry.printSchema()
latest_telemetry.show(10, truncate=False)

Latest telemetry records: 100
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)

+-------------------+---------+----------------+----------------+----------------+----------------+
|datetime           |machineID|volt            |rotate          |pressure        |vibration       |
+-------------------+---------+----------------+----------------+----------------+----------------+
|2016-01-01 06:00:00|1        |191.873171142148|382.736625649486|100.893691098222|37.9402195210172|
|2016-01-01 06:00:00|2        |182.021908044284|392.270187070234|99.9462932865942|41.6761841139842|
|2016-01-01 06:00:00|3        |164.906466435521|410.742074763206|91.7215929669456|35.4769704962203|
|2016-01-01 06:00:00|4        |149.409695703748|522.123117483713|112.680498613138|45.6378696912759|
|2016-01-01 06:00:00|5  

In [0]:
from pyspark.sql.functions import collect_set, array_contains, when

latest_with_errors = (
    latest_telemetry.alias("t")
    .join(
        errors.alias("e"),
        (col("t.machineID") == col("e.machineID")) &
        (col("e.datetime") < col("t.datetime")) &
        (col("e.datetime") >= col("t.datetime") - expr("INTERVAL 48 HOURS")),
        "left"
    )
    .groupBy(
        "t.datetime",
        "t.machineID",
        "t.volt",
        "t.rotate",
        "t.pressure",
        "t.vibration"
    )
    .agg(
        collect_set("e.errorID").alias("error_ids")
    )
    .withColumn(
        "has_error1",
        when(array_contains("error_ids", "error1"), 1).otherwise(0)
    )
    .withColumn(
        "has_error2",
        when(array_contains("error_ids", "error2"), 1).otherwise(0)
    )
    .withColumn(
        "has_error3",
        when(array_contains("error_ids", "error3"), 1).otherwise(0)
    )
    .withColumn(
        "has_error4",
        when(array_contains("error_ids", "error4"), 1).otherwise(0)
    )
    .withColumn(
        "has_error5",
        when(array_contains("error_ids", "error5"), 1).otherwise(0)
    )
)

print("Latest machine records with error features:", latest_with_errors.count())

latest_with_errors.printSchema()

latest_with_errors.show(10, truncate=False)

Latest machine records with error features: 100
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- error_ids: array (nullable = false)
 |    |-- element: string (containsNull = false)
 |-- has_error1: integer (nullable = false)
 |-- has_error2: integer (nullable = false)
 |-- has_error3: integer (nullable = false)
 |-- has_error4: integer (nullable = false)
 |-- has_error5: integer (nullable = false)

+-------------------+---------+----------------+----------------+----------------+----------------+----------------+----------+----------+----------+----------+----------+
|datetime           |machineID|volt            |rotate          |pressure        |vibration       |error_ids       |has_error1|has_error2|has_error3|has_error4|has_error5|
+-------------------+---------+----------------+----

In [0]:
prediction_pd = (
    latest_with_errors
    .select(
        "datetime",
        "machineID",
        "volt",
        "rotate",
        "pressure",
        "vibration",
        "has_error1",
        "has_error2",
        "has_error3",
        "has_error4",
        "has_error5"
    )
    .toPandas()
)

print("Prediction records:", len(prediction_pd))
print("Prediction columns:")
print(prediction_pd.columns.tolist())

prediction_pd.head(10)

Prediction records: 100
Prediction columns:
['datetime', 'machineID', 'volt', 'rotate', 'pressure', 'vibration', 'has_error1', 'has_error2', 'has_error3', 'has_error4', 'has_error5']


,datetime,machineID,volt,rotate,pressure,vibration,has_error1,has_error2,has_error3,has_error4,has_error5
0,2016-01-01 06:00:00,94,171.397961,560.595202,117.687661,39.191912,0,0,1,1,0
1,2016-01-01 06:00:00,64,176.655123,482.810554,92.312805,38.845887,1,0,0,0,0
2,2016-01-01 06:00:00,30,183.137756,429.051977,111.774259,41.118591,0,1,0,0,0
3,2016-01-01 06:00:00,95,189.171616,438.858154,87.681613,43.906112,1,0,0,0,0
4,2016-01-01 06:00:00,8,168.439184,438.340305,89.904141,32.154524,0,0,1,1,0
5,2016-01-01 06:00:00,45,158.481938,452.640928,97.041501,48.070976,0,0,1,0,0
6,2016-01-01 06:00:00,90,172.245345,356.830322,104.220166,49.848539,1,0,0,0,0
7,2016-01-01 06:00:00,23,200.553395,408.467552,100.059553,47.277976,0,0,1,0,0
8,2016-01-01 06:00:00,80,161.435535,490.037879,91.033148,45.872878,0,0,1,0,0
9,2016-01-01 06:00:00,5,178.789197,415.167298,142.414273,47.020210,0,1,0,0,0


In [0]:
# Features used by the trained Random Forest
X_current = prediction_pd[feature_columns]

# Predict probabilities for all four components
current_probabilities = model.predict_proba(X_current)

# Add probability for each component
for i, component in enumerate(model.classes_):
    prediction_pd[f"prob_{component}"] = current_probabilities[:, i]

# Select the component with the highest probability
prediction_pd["predicted_component"] = model.classes_[
    current_probabilities.argmax(axis=1)
]

# Probability of the predicted component
prediction_pd["prediction_probability"] = current_probabilities.max(axis=1)

print("Component predictions generated successfully.")

prediction_pd[
    [
        "machineID",
        "datetime",
        "predicted_component",
        "prediction_probability",
        "prob_comp1",
        "prob_comp2",
        "prob_comp3",
        "prob_comp4"
    ]
].head(10)

Component predictions generated successfully.


,machineID,datetime,predicted_component,prediction_probability,prob_comp1,prob_comp2,prob_comp3,prob_comp4
0,94,2016-01-01 06:00:00,comp3,0.876997,0.050000,0.068003,0.876997,0.005000
1,64,2016-01-01 06:00:00,comp1,1.000000,1.000000,0.000000,0.000000,0.000000
2,30,2016-01-01 06:00:00,comp2,0.395000,0.241864,0.395000,0.218136,0.145000
3,95,2016-01-01 06:00:00,comp1,0.996550,0.996550,0.000000,0.000000,0.003450
4,8,2016-01-01 06:00:00,comp3,0.410701,0.216684,0.302615,0.410701,0.070000
5,45,2016-01-01 06:00:00,comp4,0.335000,0.245000,0.310000,0.110000,0.335000
6,90,2016-01-01 06:00:00,comp1,0.908794,0.908794,0.010000,0.007124,0.074083
7,23,2016-01-01 06:00:00,comp1,0.477148,0.477148,0.280000,0.072852,0.170000
8,80,2016-01-01 06:00:00,comp2,0.330000,0.310000,0.330000,0.120000,0.240000
9,5,2016-01-01 06:00:00,comp3,0.361698,0.119231,0.344071,0.361698,0.175000


In [0]:
gold_q7_pd = prediction_pd[
    [
        "machineID",
        "datetime",
        "volt",
        "rotate",
        "pressure",
        "vibration",
        "has_error1",
        "has_error2",
        "has_error3",
        "has_error4",
        "has_error5",
        "prob_comp1",
        "prob_comp2",
        "prob_comp3",
        "prob_comp4",
        "predicted_component",
        "prediction_probability"
    ]
].copy()

print("Gold Q7 records:", len(gold_q7_pd))
gold_q7_pd.head(10)

Gold Q7 records: 100


,machineID,datetime,volt,rotate,pressure,vibration,has_error1,has_error2,has_error3,has_error4,has_error5,prob_comp1,prob_comp2,prob_comp3,prob_comp4,predicted_component,prediction_probability
0,94,2016-01-01 06:00:00,171.397961,560.595202,117.687661,39.191912,0,0,1,1,0,0.050000,0.068003,0.876997,0.005000,comp3,0.876997
1,64,2016-01-01 06:00:00,176.655123,482.810554,92.312805,38.845887,1,0,0,0,0,1.000000,0.000000,0.000000,0.000000,comp1,1.000000
2,30,2016-01-01 06:00:00,183.137756,429.051977,111.774259,41.118591,0,1,0,0,0,0.241864,0.395000,0.218136,0.145000,comp2,0.395000
3,95,2016-01-01 06:00:00,189.171616,438.858154,87.681613,43.906112,1,0,0,0,0,0.996550,0.000000,0.000000,0.003450,comp1,0.996550
4,8,2016-01-01 06:00:00,168.439184,438.340305,89.904141,32.154524,0,0,1,1,0,0.216684,0.302615,0.410701,0.070000,comp3,0.410701
5,45,2016-01-01 06:00:00,158.481938,452.640928,97.041501,48.070976,0,0,1,0,0,0.245000,0.310000,0.110000,0.335000,comp4,0.335000
6,90,2016-01-01 06:00:00,172.245345,356.830322,104.220166,49.848539,1,0,0,0,0,0.908794,0.010000,0.007124,0.074083,comp1,0.908794
7,23,2016-01-01 06:00:00,200.553395,408.467552,100.059553,47.277976,0,0,1,0,0,0.477148,0.280000,0.072852,0.170000,comp1,0.477148
8,80,2016-01-01 06:00:00,161.435535,490.037879,91.033148,45.872878,0,0,1,0,0,0.310000,0.330000,0.120000,0.240000,comp2,0.330000
9,5,2016-01-01 06:00:00,178.789197,415.167298,142.414273,47.020210,0,1,0,0,0,0.119231,0.344071,0.361698,0.175000,comp3,0.361698


In [0]:
gold_q7 = spark.createDataFrame(gold_q7_pd)

print("Gold Q7 Spark DataFrame created.")
gold_q7.printSchema()

Gold Q7 Spark DataFrame created.
root
 |-- machineID: integer (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- has_error1: integer (nullable = true)
 |-- has_error2: integer (nullable = true)
 |-- has_error3: integer (nullable = true)
 |-- has_error4: integer (nullable = true)
 |-- has_error5: integer (nullable = true)
 |-- prob_comp1: double (nullable = true)
 |-- prob_comp2: double (nullable = true)
 |-- prob_comp3: double (nullable = true)
 |-- prob_comp4: double (nullable = true)
 |-- predicted_component: string (nullable = true)
 |-- prediction_probability: double (nullable = true)



In [0]:
from delta.tables import DeltaTable

target_table = "workspace.default.gold_q7_component_prediction"

if spark.catalog.tableExists(target_table):

    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("t")
        .merge(
            gold_q7.alias("s"),
            "t.machineID = s.machineID AND t.datetime = s.datetime"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Gold Q7 table updated using MERGE.")

else:

    (
        gold_q7.write
        .format("delta")
        .saveAsTable(target_table)
    )

    print("Gold Q7 table created.")

gold_q7.printSchema()

Gold Q7 table created.
root
 |-- machineID: integer (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- has_error1: integer (nullable = true)
 |-- has_error2: integer (nullable = true)
 |-- has_error3: integer (nullable = true)
 |-- has_error4: integer (nullable = true)
 |-- has_error5: integer (nullable = true)
 |-- prob_comp1: double (nullable = true)
 |-- prob_comp2: double (nullable = true)
 |-- prob_comp3: double (nullable = true)
 |-- prob_comp4: double (nullable = true)
 |-- predicted_component: string (nullable = true)
 |-- prediction_probability: double (nullable = true)



In [0]:
q7_gold = spark.table(
    "workspace.default.gold_q7_component_prediction"
)

print("Q7 Gold record count:", q7_gold.count())

q7_gold.orderBy(
    col("prediction_probability").desc()
).show(10, truncate=False)

Q7 Gold record count: 100
+---------+-------------------+----------------+----------------+----------------+----------------+----------+----------+----------+----------+----------+-------------------+--------------------+--------------------+---------------------+-------------------+----------------------+
|machineID|datetime           |volt            |rotate          |pressure        |vibration       |has_error1|has_error2|has_error3|has_error4|has_error5|prob_comp1         |prob_comp2          |prob_comp3          |prob_comp4           |predicted_component|prediction_probability|
+---------+-------------------+----------------+----------------+----------------+----------------+----------+----------+----------+----------+----------+-------------------+--------------------+--------------------+---------------------+-------------------+----------------------+
|64       |2016-01-01 06:00:00|176.655123080909|482.810553929662|92.3128052155236|38.8458866823918|1         |0         |0      

In [0]:
component_prediction_counts = (
    q7_gold
    .groupBy("predicted_component")
    .count()
    .orderBy("predicted_component")
)

display(component_prediction_counts)

predicted_component,count
comp1,76
comp2,4
comp3,10
comp4,10


Databricks visualization. Run in Databricks to view.